# WordPiece Tokenization and BERT Pooling
Pure NumPy implementation of greedy WordPiece tokenization, CLS pooling, and padding-aware mean pooling.

In [1]:
import numpy as np

vocab = {"play", "##ing", "the", "cat", "##s", "un", "##aff", "##able"}
embeddings = np.arange(12, dtype=float).reshape(4, 3)
mask = np.array([1, 1, 0, 1])

In [2]:
def wordpiece_tokenize(word, vocab, unk_token="[UNK]"):
    pieces = []
    start = 0

    while start < len(word):
        end = len(word)
        matched_piece = None
        matched_end = None

        while end > start:
            substring = word[start:end]
            candidate = substring if start == 0 else "##" + substring

            if candidate in vocab:
                matched_piece = candidate
                matched_end = end
                break
            end -= 1

        if matched_piece is None:
            return [unk_token]

        pieces.append(matched_piece)
        start = matched_end

    return pieces

In [3]:
def tokenize_text(text, vocab, unk_token="[UNK]"):
    tokens = []
    for word in text.split():
        tokens.extend(wordpiece_tokenize(word, vocab, unk_token))
    return tokens

In [4]:
def cls_pool(embeddings):
    embeddings = np.asarray(embeddings)
    return embeddings[0]

In [5]:
def mean_pool(embeddings, mask):
    embeddings = np.asarray(embeddings, dtype=float)
    mask = np.asarray(mask)
    real_tokens = mask == 1

    if not np.any(real_tokens):
        return np.zeros(embeddings.shape[-1], dtype=float)

    return embeddings[real_tokens].mean(axis=0)

In [6]:
test_vocab = {"play", "##ing", "the", "cat", "##s"}
print(wordpiece_tokenize("playing", test_vocab))
print(wordpiece_tokenize("zzz", test_vocab))
print(tokenize_text("the cats", test_vocab))

E = np.arange(12, dtype=float).reshape(4, 3)
print(cls_pool(E))
print(mean_pool(E, np.array([1, 1, 0, 1])))

['play', '##ing']
['[UNK]']
['the', 'cat', '##s']
[0. 1. 2.]
[4. 5. 6.]
